In [0]:
df=spark.read.table("uberxxxxxx000000.bronze.bulk_rides")

In [0]:
df.schema

In [0]:
df=spark.read.table("uberxxxxxx000000.bronze.rides_raw")

In [0]:
df.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType

# 1. Define your exact target schema based on your schema printout
rides_schema = StructType([
    StructField("base_fare", DoubleType(), True),
    StructField("booking_timestamp", StringType(), True),
    StructField("cancellation_reason_id", LongType(), True),
    StructField("confirmation_number", StringType(), True),
    StructField("distance_fare", DoubleType(), True),
    StructField("distance_miles", DoubleType(), True),
    StructField("driver_id", StringType(), True),
    StructField("driver_license", StringType(), True),
    StructField("driver_name", StringType(), True),
    StructField("driver_phone", StringType(), True),
    StructField("driver_rating", DoubleType(), True),
    StructField("dropoff_address", StringType(), True),
    StructField("dropoff_city_id", LongType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("dropoff_location_id", StringType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("dropoff_timestamp", StringType(), True),
    StructField("duration_minutes", LongType(), True),
    StructField("license_plate", StringType(), True),
    StructField("passenger_email", StringType(), True),
    StructField("passenger_id", StringType(), True),
    StructField("passenger_name", StringType(), True),
    StructField("passenger_phone", StringType(), True),
    StructField("payment_method_id", LongType(), True),
    StructField("pickup_address", StringType(), True),
    StructField("pickup_city_id", LongType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("pickup_location_id", StringType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("pickup_timestamp", StringType(), True),
    StructField("rating", LongType(), True),
    StructField("ride_id", StringType(), True),
    StructField("ride_status_id", LongType(), True),
    StructField("subtotal", DoubleType(), True),
    StructField("surge_multiplier", DoubleType(), True),
    StructField("time_fare", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("total_fare", DoubleType(), True),
    StructField("vehicle_color", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("vehicle_make_id", LongType(), True),
    StructField("vehicle_model", StringType(), True),
    StructField("vehicle_type_id", LongType(), True),
])

# 2. Parse the string JSON column using from_json and flatten it with '*'
parsed_df =  df.withColumn(
    "parsed_json", F.from_json(F.col("rides"), rides_schema)
).select("parsed_json.*")
# Display the fully expanded dataframe matching your schema columns


In [0]:
parsed_df.display()

In [0]:
%sql
select count(*) from uberxxxxxx000000.bronze.stg_rides limit 5

###Jinja TEmplate
``

In [0]:
pip install Jinja2

In [0]:
jinja_config=[
    {
        "table":"uberxxxxxx000000.bronze.stg_rides",
        "alias":"stg",
        "select":"stg.*",
        "where":""
    },
    {
        "table":"uberxxxxxx000000.bronze.map_vehicle_makes",
        "alias":"mvm",
        "select":"mvm.vehicle_make",
        "where":"",
        "on" : "stg.vehicle_make_id = mvm.vehicle_make_id"
        
    },

    {
        "table":"uberxxxxxx000000.bronze.map_vehicle_types",
        "alias":"mvt",
        "select":"mvt.vehicle_type,mvt.description,mvt.base_rate,mvt.per_mile,mvt.per_minute",
        
        "where":"",
        "on" : "stg.vehicle_type_id = mvt.vehicle_type_id"
        
    }
    
]

In [0]:
from jinja2 import Template

sql_template = """
SELECT 
    {%- for conf in jinja_config %}
    {{ conf.select }}{% if not loop.last %},{% endif %}
    {%- endfor %}
FROM {{ jinja_config[0].table }} AS {{ jinja_config[0].alias }}
{%- for conf in jinja_config[1:] %}
LEFT JOIN {{ conf.table }} AS {{ conf.alias }} ON {{ conf.on }}
    {%- if conf.where %}
    WHERE {{ conf.where }}
    {%- endif %}
{%- endfor %}
"""

# Render the template using your configuration array
template = Template(sql_template)
rendered_sql = template.render(jinja_config=jinja_config)

print(rendered_sql)



In [0]:
df=spark.sql(rendered_sql)


In [0]:
%sql
select * from uberxxxxxx000000.bronze.map_cities limit 2

In [0]:
df=spark.read.table("uberxxxxxx000000.bronze.bulk_rides")

In [0]:
from pyspark.sql.functions import *

In [0]:
df=df.withColumn("booking_timestamp",col("booking_timestamp").cast("timestamp"))


In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("uberxxxxxx000000.bronze.bulk_rides")


In [0]:
%sql
drop table uberxxxxxx000000.bronze.stg_rides

In [0]:
%sql
select * from uberxxxxxx000000.bronze.stg_rides limit 2

In [0]:
%sql
select * from uberxxxxxx000000.bronze.silver_obt

In [0]:
%sql
drop table uberxxxxxx000000.bronze.silver_obt

In [0]:
%sql
drop table uberxxxxxx000000.silver.intial_rides

In [0]:
%sql
select * from uberxxxxxx000000.silver.silver_obt limit 2

In [0]:
%sql
select 
passenger_id,
passenger_email,
passenger_name,
passenger_phone
from uberxxxxxx000000.silver.silver_obt


In [0]:
%sql
select * from uberxxxxxx000000.gold.dim_passenger_scd2

In [0]:
%sql
update uberxxxxxx000000.gold.dim_passenger_scd2 set passenger_email='test@gmail.com' where passenger_id='06aa7425-bd51-405d-9592-9ff2bda13fd1'

In [0]:
%sql
select * from uberxxxxxx000000.gold.dim_passenger_scd2 where passenger_id='06aa7425-bd51-405d-9592-9ff2bda13fd1'

In [0]:
%sql
INSERT INTO uberxxxxxx000000.bronze.bulk_rides
SELECT
    uuid()            AS ride_id,
    confirmation_number, passenger_id, driver_id, vehicle_id,
    pickup_location_id, dropoff_location_id, vehicle_type_id, vehicle_make_id,
    payment_method_id, ride_status_id, pickup_city_id, dropoff_city_id,
    cancellation_reason_id, passenger_name,
    'test@gmail.com'  AS passenger_email,
    passenger_phone, driver_name, driver_rating, driver_phone, driver_license,
    vehicle_model, vehicle_color, license_plate, pickup_address, pickup_latitude,
    pickup_longitude, dropoff_address, dropoff_latitude, dropoff_longitude,
    distance_miles, duration_minutes,
    current_timestamp() AS booking_timestamp,
    pickup_timestamp, dropoff_timestamp, base_fare, distance_fare, time_fare,
    surge_multiplier, subtotal, tip_amount, total_fare, rating
FROM uberxxxxxx000000.bronze.stg_rides
WHERE passenger_id = '06aa7425-bd51-405d-9592-9ff2bda13fd1'
LIMIT 1;

In [0]:
%sql
INSERT INTO uberxxxxxx000000.bronze.bulk_rides BY NAME
SELECT
    uuid() AS ride_id,
    confirmation_number, passenger_id, driver_id, vehicle_id,
    pickup_location_id, dropoff_location_id, vehicle_type_id, vehicle_make_id,
    payment_method_id, ride_status_id, pickup_city_id, dropoff_city_id,
    cancellation_reason_id, passenger_name,
    'test@gmail.com' AS passenger_email,
    passenger_phone, driver_name, driver_rating, driver_phone, driver_license,
    vehicle_model, vehicle_color, license_plate, pickup_address, pickup_latitude,
    pickup_longitude, dropoff_address, dropoff_latitude, dropoff_longitude,
    distance_miles, duration_minutes,
    current_timestamp() AS booking_timestamp,
    pickup_timestamp, dropoff_timestamp, base_fare, distance_fare, time_fare,
    surge_multiplier, subtotal, tip_amount, total_fare, rating
FROM uberxxxxxx000000.bronze.stg_rides
WHERE passenger_id = '06aa7425-bd51-405d-9592-9ff2bda13fd1'
LIMIT 1;

In [0]:
%sql
select * from uberxxxxxx000000.silver.silver_obt limit 1